In [0]:
##Read data from Azure Data Lake

import urllib.request

url = "https://bb49.s3.ap-south-1.amazonaws.com/data.txt"

user = spark.sql("SELECT current_user()").collect()[0][0]

path = f"/Workspace/Users/{user}/data.txt"

urllib.request.urlretrieve(url, path)

df = spark.read.option("header", "true").option("inferSchema", "true").parquet("file:" + path)


df.show()
df.printSchema()





+-------------+-----+-----+----------+
|     username|value|score|regioncode|
+-------------+-----+-----+----------+
|beautifulbear|   10|   55|        45|
|beautifulbear|   10|   53|        01|
|beautifulbear|   01|   12|        33|
|beautifulbear|   13|   48|        30|
|beautifulbear|   07|   19|        30|
|beautifulbear|   16|   19|        00|
|beautifulbear|   00|   02|        00|
|beautifulbear|   13|   00|        30|
|beautifulbear|   23|   13|        53|
|beautifulbear|   22|   03|        01|
|beautifulbear|   10|   17|        48|
|beautifulbear|   05|   25|        00|
|beautifulbear|   01|   41|        00|
|beautifulbear|   00|   50|        00|
|beautifulbear|   23|   44|        30|
|beautifulbear|   04|   58|        30|
|beautifulbear|   03|   20|        30|
|beautifulbear|   13|   08|        30|
|beautifulbear|   17|   00|        37|
|beautifulbear|   19|   16|        00|
+-------------+-----+-----+----------+
only showing top 20 rows
root
 |-- username: string (nullable = 

In [0]:

##Transforming the data - collecting the score for each username

from pyspark.sql.functions import *

aggdf = df.groupBy("username")\
    .agg(collect_list("score"))\
        .withColumnRenamed("collect_list(score)", "scores")

aggdf.show()

aggdf.printSchema()


# aggdf.select(
#     "username",
#     slice(col("collect_list(score)"), 1, 10).alias("scores_preview"),  # first 10 elements
#     size(col("collect_list(score)")).alias("total_count")               # so you know how many were cut off
# ).show(truncate=False)

+-----------------+--------------------+
|         username|              scores|
+-----------------+--------------------+
|    beautifulwolf|[38, 01, 47, 25, ...|
|  beautifulrabbit|[03, 12, 53, 52, ...|
|     bigbutterfly|[22, 50, 28, 16, ...|
|        blacklion|[40, 34, 09, 56, ...|
|     blackmeercat|[21, 39, 50, 29, ...|
|         bigkoala|[22, 13, 03, 32, ...|
|         bigtiger|[39, 27, 11, 53, ...|
|         blackcat|[20, 36, 14, 29, ...|
|     blackleopard|[41, 45, 07, 47, ...|
| beautifulladybug|[47, 45, 40, 45, ...|
|    beautifulduck|[27, 39, 06, 48, ...|
|beautifulelephant|[24, 39, 57, 34, ...|
|      bigelephant|[20, 19, 43, 15, ...|
|       bigladybug|[59, 21, 20, 52, ...|
|          bigduck|[57, 51, 32, 01, ...|
|    beautifulbird|[53, 02, 20, 29, ...|
| beautifulmeercat|[22, 02, 12, 34, ...|
|   beautifulsnake|[28, 58, 51, 19, ...|
|         bigpanda|[56, 18, 06, 48, ...|
|         bigsnake|[38, 44, 35, 20, ...|
+-----------------+--------------------+
only showing top

In [0]:
%sql

-- Creating a lakehouse db using sql


create database if not exists zeyodb




In [0]:
# Writing the transformed data to lakehouse table using pyspark

aggdf.write.mode("overwrite")\
    .saveAsTable("zeyodb.adl_tab")


In [0]:
%sql

--validating the lakehouse table write

select * from zeyodb.adl_tab limit 20

username scores beautifulwolf List(38, 01, 47, 25, 55, 28, 00, 42, 16, 34, 30, 23, 43, 38, 14, 18, 05, 50, 14, 23, 17, 02, 00, 25, 27, 48, 29, 36, 48, 23, 45, 55, 10, 38, 42, 31, 22, 56, 43, 25, 00, 31, 03, 27, 35, 55, 14, 23, 05, 31, 03, 25, 35, 10, 45, 07, 58, 49, 09, 00, 23, 14, 19, 17, 03, 45, 23, 20, 53, 59, 32, 38, 30, 09, 21, 42, 58, 15, 15, 42, 49, 49, 25, 30, 30, 43, 14, 25, 34, 09, 36, 13, 12, 20, 35, 51, 21, 07, 56, 02, 05, 16, 38, 37, 47, 35, 49, 58, 53, 14, 45, 18, 49, 06, 20, 37, 56, 36, 42, 00, 06, 03, 21, 09, 21, 41, 39, 01, 42, 43, 48, 23, 53, 07, 43, 03, 52, 31, 01, 58, 46, 42, 25, 08, 33, 42, 12, 30, 16, 28, 33, 19, 42, 06, 31, 02, 46, 01, 01, 43, 43, 11, 00, 36, 53, 42, 14, 50, 46, 26, 38, 30, 11, 58, 22, 09, 51, 08, 10, 15, 49, 01, 57, 45, 00, 00, 43, 53, 45, 53, 22, 50, 01, 50, 49, 55, 57, 47, 36, 02, 15, 45, 50, 48, 33, 20, 53, 10, 27, 19, 08, 30, 06, 25, 50, 28, 23, 08, 39, 07, 49, 51, 23, 16, 30, 56, 33, 30, 33, 04, 31, 43, 23, 31, 46, 48, 38, 34, 24, 45, 00, 51, 12, 23, 29, 45, 23, 50, 35, 29, 04, 53, 23, 56, 26, 10, 21, 03, 51, 55, 27, 53, 23, 31, 56, 48, 26, 02, 41, 06, 11, 17, 01, 18, 01, 22, 36, 55, 46, 24, 50, 57, 11, 06, 40, 02, 08, 09, 04, 52, 19, 16, 34, 46, 07, 57, 16, 26, 32, 52, 51, 37, 33, 36, 12, 54, 46, 00, 20, 17, 34, 32, 09, 10, 17, 22, 13, 05, 26, 54, 54, 28, 06, 19, 31, 12, 35, 56, 01, 02, 54, 24, 05, 50, 00, 01, 45, 03, 48, 40, 10, 06, 18, 52, 58, 25, 23, 25, 52, 31, 08, 46, 28, 35, 44, 06, 35, 35, 05, 01, 01, 24, 18, 49, 09, 39, 00, 13, 41, 06, 49, 21, 35, 28, 34, 57, 06, 17, 39, 43, 41, 25, 08, 32, 23, 31, 30, 00, 00, 19, 47, 37, 48, 23, 17, 22, 08, 02, 03, 46, 55, 33, 21, 08, 26, 41, 38, 07, 33, 23, 21, 22, 49, 15, 57, 22, 04, 02, 11, 16, 55, 43, 14, 08, 57, 14, 58, 08, 40, 57, 33, 01, 07, 00, 35, 50, 20, 24, 46, 29, 47, 12, 08, 20, 46, 07, 01, 50, 50, 02, 31, 39, 50, 57, 20, 02, 50, 28, 15, 49, 39, 28, 20, 49, 39, 03, 41, 01, 26, 50, 56, 59, 04, 51, 34, 43, 28, 35, 23, 28, 48, 04, 35, 25, 34, 43, 23, 08, 02, 10, 40, 28, 29, 06, 13, 01, 15, 02, 00, 06, 02, 46, 31, 49, 38, 04, 14, 57, 03, 41, 32, 36, 31, 22, 34, 59, 52, 50, 24, 10, 51, 59, 58, 04, 22, 15, 13, 07, 03, 12, 12, 00, 12, 38, 44, 54, 14, 00, 49, 15, 58, 05, 30, 48, 29, 01, 22, 59, 48, 00, 30, 28, 50, 19, 22, 43, 34, 35, 27, 12, 11, 12, 13, 17, 38, 52, 57, 18, 05, 44, 11, 37, 44, 20, 07, 07, 22, 24, 47, 17, 41, 57, 12, 54, 52, 16, 12, 25, 57, 37, 41, 53, 31, 55, 30, 12, 33, 23, 29, 00, 07, 11, 49, 43, 20, 26, 16, 47, 04, 01, 01, 26, 25, 41, 12, 58, 27, 46, 32, 02, 28, 44, 30, 38, 02, 21, 20, 35, 22, 29, 05, 33, 17, 47, 24, 55, 45, 15, 30, 00, 00, 58, 50, 50, 53, 16, 03, 51, 16, 40, 48, 08, 54, 00, 42, 39, 19, 52, 01, 29, 16, 35, 16, 21, 56, 47, 35, 22, 49, 51, 43, 28, 47, 48, 54, 01, 08, 48, 17, 33, 23, 59, 32, 42, 07, 55, 05, 57, 15, 22, 47, 29, 45, 57, 58, 15, 49, 47, 33, 52, 45, 14, 38, 49, 22, 20, 29, 41, 09, 52, 56, 51, 51, 00, 33, 56, 17, 40, 38, 31, 56, 25, 50, 39, 11, 52, 02, 14, 24, 14, 25, 45, 07, 23, 01, 47, 26, 35, 17, 37, 02, 20, 26, 22, 54, 42, 49, 11, 32, 23, 19, 31, 16, 06, 09, 20, 07, 57, 24, 17, 45, 31, 48, 00, 24, 35, 39, 21, 40, 31, 21, 55, 24, 52, 54, 31, 53, 58, 16, 30, 33, 50, 53, 49, 06, 45, 32, 26, 47, 11, 53, 30, 27, 14, 56, 21, 57, 18, 40, 20, 53, 00, 01, 42, 14, 48, 15, 45, 42, 23, 50, 50, 43, 10, 24, 44, 01, 22, 31, 45, 31, 37, 22, 03, 34, 08, 23, 54, 03, 12, 53, 06, 50, 12, 19, 02, 38, 52, 47, 53, 30, 54, 26, 26, 05, 13, 27, 03, 30, 07, 54, 19, 19, 52, 50, 24, 33, 33, 33, 41, 00, 47, 09, 32, 41, 20, 04, 13, 37, 39, 22, 11, 39, 39, 04, 21, 19, 09, 02, 20, 50, 48, 53, 59, 28, 39, 15, 54, 17, 37, 10, 55, 41, 26, 13, 51, 41, 24, 30, 07, 26, 48, 05, 48, 30, 42, 16, 17, 07, 50, 57, 52, 12, 39, 24, 20, 08, 32, 48, 25, 32, 04, 01, 34, 23, 20, 51, 37, 58, 36, 41, 43, 00, 46, 15, 13, 22, 56, 58, 24, 13, 24, 46, 47, 29, 08, 13, 25, 00, 14, 22, 35, 19, 00, 59, 57, 20, 23, 32, 42, 54, 13, 49, 20, 54, 11, 59, 30, 42, 08, 34, 22, 24, 23, 31, 30, 30, 49, 03, 32, 13, 29, 46, 13, 03, 54, 27, 23, 07, 58, 01, 2